# Day 4: pandas as SQL

One month of LT day-ahead prices (October 2026, includes the 25 Oct clock change), 10 SQL -> pandas exercises.

In [1]:
import pandas as pd
import numpy as np
from datetime import date, datetime, timezone
import httpx
from lt_grid_pulse.utils.timez import local_day_bounds_utc, intervals_in_local_day, VILNIUS
from lt_grid_pulse.ingest.elering import fetch_elering_api_nps_controller, parse_and_filter, daily_min_max_avg_row_cnt

In [2]:
# Exercise 1: load one month, parse timestamps, filter to the window
# SQL: SELECT DATEADD(second, timestamp, '1970-01-01') AT TIME ZONE 'UTC' AS ts_utc, price
#      FROM elering_lt
#      WHERE ts_utc >= @start AND ts_utc < @end      -- API's end is inclusive, so the filter drops 1 row
BASE_URL = "https://dashboard.elering.ee/api/nps/price"
LOCATION_CODE = "lt"
START_DATE = date(2026, 10, 1)
END_DATE = date(2026, 10, 31)

start_dt = local_day_bounds_utc(START_DATE)[0]
end_dt = local_day_bounds_utc(END_DATE)[1]

params = {"start": start_dt.isoformat(), "end": end_dt.isoformat()}

r = httpx.get(BASE_URL, params=params, timeout=3)
print(r.status_code)

r_json = r.json()
print(r_json.get("success"))

data = r_json["data"][LOCATION_CODE]
data_df = pd.DataFrame(data)
print("raw rows:", len(data_df))                     # 2981
data_df["ts_utc"] = pd.to_datetime(data_df["timestamp"], unit="s", utc=True)
data_df = data_df[(start_dt <= data_df["ts_utc"]) & (data_df["ts_utc"] < end_dt)]
data_df = data_df[["ts_utc", "price"]]
print("rows after filter:", len(data_df))            # 2980
print(data_df.dtypes)

200
True
raw rows: 2981
rows after filter: 2980
ts_utc    datetime64[us, UTC]
price                 float64
dtype: object


In [3]:
# Exercise 2: derived local-time columns
# SQL: SELECT *, ts_utc AT TIME ZONE 'FLE Standard Time' AS ts_local,
#             CAST(ts_local AS date) AS date_local, DATEPART(hour, ts_local) AS hour_local,
#             DATEPART(weekday, ts_local) AS weekday
#      FROM #prices
data_df["ts_local"] = data_df["ts_utc"].dt.tz_convert(VILNIUS)
data_df["date_local"] = data_df["ts_local"].dt.date
data_df["hour_local"] = data_df["ts_local"].dt.hour
data_df["weekday"] = data_df["ts_local"].dt.day_of_week
print(data_df)

                        ts_utc   price                  ts_local  date_local  \
0    2026-09-30 21:00:00+00:00   85.83 2026-10-01 00:00:00+03:00  2026-10-01   
1    2026-09-30 21:15:00+00:00   39.92 2026-10-01 00:15:00+03:00  2026-10-01   
2    2026-09-30 21:30:00+00:00   30.66 2026-10-01 00:30:00+03:00  2026-10-01   
3    2026-09-30 21:45:00+00:00   27.80 2026-10-01 00:45:00+03:00  2026-10-01   
4    2026-09-30 22:00:00+00:00   56.93 2026-10-01 01:00:00+03:00  2026-10-01   
...                        ...     ...                       ...         ...   
2975 2026-10-31 20:45:00+00:00  238.02 2026-10-31 22:45:00+02:00  2026-10-31   
2976 2026-10-31 21:00:00+00:00  232.19 2026-10-31 23:00:00+02:00  2026-10-31   
2977 2026-10-31 21:15:00+00:00  204.28 2026-10-31 23:15:00+02:00  2026-10-31   
2978 2026-10-31 21:30:00+00:00  199.51 2026-10-31 23:30:00+02:00  2026-10-31   
2979 2026-10-31 21:45:00+00:00  198.42 2026-10-31 23:45:00+02:00  2026-10-31   

      hour_local  weekday  
0          

In [4]:
# Exercise 3: WHERE + COUNT
# SQL: SELECT COUNT(*) FROM #prices WHERE price < 0;
#      SELECT COUNT(*) FROM #prices WHERE price > 200;
print((data_df["price"] < 0).sum())
print((data_df["price"] > 200).sum())

21
648


In [5]:
# Exercise 4: TOP 10 priciest intervals
# SQL: SELECT TOP 10 ts_local, price FROM #prices ORDER BY price DESC
print(data_df[["ts_local", "price"]].sort_values("price", ascending=False).head(10))
print(data_df[["ts_local", "price"]].nlargest(10, "price"))

                      ts_local   price
2767 2026-10-29 18:45:00+02:00  468.48
2772 2026-10-29 20:00:00+02:00  458.87
2763 2026-10-29 17:45:00+02:00  442.91
2762 2026-10-29 17:30:00+02:00  422.91
2723 2026-10-29 07:45:00+02:00  406.70
1217 2026-10-13 16:15:00+03:00  400.01
2773 2026-10-29 20:15:00+02:00  400.00
2759 2026-10-29 16:45:00+02:00  400.00
2751 2026-10-29 14:45:00+02:00  399.17
2730 2026-10-29 09:30:00+02:00  398.32
                      ts_local   price
2767 2026-10-29 18:45:00+02:00  468.48
2772 2026-10-29 20:00:00+02:00  458.87
2763 2026-10-29 17:45:00+02:00  442.91
2762 2026-10-29 17:30:00+02:00  422.91
2723 2026-10-29 07:45:00+02:00  406.70
1217 2026-10-13 16:15:00+03:00  400.01
2759 2026-10-29 16:45:00+02:00  400.00
2773 2026-10-29 20:15:00+02:00  400.00
2751 2026-10-29 14:45:00+02:00  399.17
2730 2026-10-29 09:30:00+02:00  398.32


In [6]:
# Exercise 5a/5b: daily stats + completeness check
# SQL: SELECT date_local, COUNT(price), MIN(price), MAX(price), AVG(price) INTO #daily
#      FROM #prices GROUP BY date_local;
#      SELECT * FROM #daily WHERE count <> dbo.intervals_in_local_day(date_local)   -- expect 0 rows
data_df_daily = data_df.groupby("date_local")["price"].agg(["count", "min", "max", "mean"]).reset_index()
data_df_daily["expected_count"] = data_df_daily["date_local"].map(intervals_in_local_day)
print(data_df_daily)
data_df_daily_bad_rows = data_df_daily[data_df_daily["count"] != data_df_daily["expected_count"]]
print(data_df_daily_bad_rows)

    date_local  count     min     max        mean  expected_count
0   2026-10-01     96   10.53  162.21   51.558333              96
1   2026-10-02     96   11.23  145.77   55.554688              96
2   2026-10-03     96   20.78  164.47   46.502917              96
3   2026-10-04     96    5.00   76.80   34.540104              96
4   2026-10-05     96    0.00   24.80    7.166042              96
5   2026-10-06     96   -0.40  262.72   63.851146              96
6   2026-10-07     96  144.17  284.68  197.742917              96
7   2026-10-08     96   26.36  187.66  135.295208              96
8   2026-10-09     96   10.25   76.55   28.313021              96
9   2026-10-10     96    9.99   57.78   24.750417              96
10  2026-10-11     96   27.99  252.80  105.218542              96
11  2026-10-12     96   26.29  287.48  151.638542              96
12  2026-10-13     96  124.23  400.01  224.457812              96
13  2026-10-14     96   21.17  264.13  105.635417              96
14  2026-1

In [7]:
# Exercise 5c: reconcile 25 Oct (the 100-interval day) with the Day 3 functions
# SQL: SELECT count, min, max, mean FROM #daily WHERE date_local = '2026-10-25'
#      -- compared with: EXEC dbo.day3_daily_stats '2026-10-25'
reconsile_check_date = date(2026, 10, 25)
reconsile_check_data = fetch_elering_api_nps_controller(reconsile_check_date)
reconsile_check_data_filtered = parse_and_filter(reconsile_check_date, reconsile_check_data)
reconsile_check_data_filtered_counts = daily_min_max_avg_row_cnt(reconsile_check_data_filtered)
reconcile_counts_df = pd.DataFrame(reconsile_check_data_filtered_counts, index=[0])

df_expected_row_count = data_df_daily.loc[data_df_daily["date_local"] == reconsile_check_date, "expected_count"]
df_expected_min = data_df_daily.loc[data_df_daily["date_local"] == reconsile_check_date, "min"]
df_expected_max = data_df_daily.loc[data_df_daily["date_local"] == reconsile_check_date, "max"]
df_expected_mean = data_df_daily.loc[data_df_daily["date_local"] == reconsile_check_date, "mean"]

reconcile_counts_df_row_count = reconcile_counts_df.loc[0, "valid_price_row_cnt"]
reconcile_counts_df_row_min = reconcile_counts_df.loc[0, "min_price"]
reconcile_counts_df_row_max = reconcile_counts_df.loc[0, "max_price"]
reconcile_counts_df_row_mean = reconcile_counts_df.loc[0, "avg_price"]

print(df_expected_row_count == reconcile_counts_df_row_count)
print(df_expected_min == reconcile_counts_df_row_min)
print(df_expected_max == reconcile_counts_df_row_max)
print(df_expected_mean == reconcile_counts_df_row_mean)

24    True
Name: expected_count, dtype: bool
24    True
Name: min, dtype: bool
24    True
Name: max, dtype: bool
24    True
Name: mean, dtype: bool


In [8]:
# Exercise 6: priciest interval per day
# SQL: WITH r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY date_local ORDER BY price DESC) AS rn FROM #prices)
#      SELECT date_local, ts_local, price FROM r WHERE rn = 1
data_df["rn"] = data_df.groupby("date_local")["price"].rank(method="first", ascending=False)
top_a = data_df.loc[data_df["rn"] == 1, ["date_local", "ts_local", "price"]]

top_labels = data_df.groupby("date_local")["price"].idxmax()
top_b = data_df.loc[top_labels, ["date_local", "ts_local", "price"]]

print(len(top_a), len(top_b))
print(top_a)
print((top_a["price"].values == top_b["price"].values).all())

31 31
      date_local                  ts_local   price
36    2026-10-01 2026-10-01 09:00:00+03:00  162.21
170   2026-10-02 2026-10-02 18:30:00+03:00  145.77
280   2026-10-03 2026-10-03 22:00:00+03:00  164.47
364   2026-10-04 2026-10-04 19:00:00+03:00   76.80
428   2026-10-05 2026-10-05 11:00:00+03:00   24.80
557   2026-10-06 2026-10-06 19:15:00+03:00  262.72
612   2026-10-07 2026-10-07 09:00:00+03:00  284.68
708   2026-10-08 2026-10-08 09:00:00+03:00  187.66
857   2026-10-09 2026-10-09 22:15:00+03:00   76.55
944   2026-10-10 2026-10-10 20:00:00+03:00   57.78
1031  2026-10-11 2026-10-11 17:45:00+03:00  252.80
1125  2026-10-12 2026-10-12 17:15:00+03:00  287.48
1217  2026-10-13 2026-10-13 16:15:00+03:00  400.01
1336  2026-10-14 2026-10-14 22:00:00+03:00  264.13
1411  2026-10-15 2026-10-15 16:45:00+03:00  329.02
1476  2026-10-16 2026-10-16 09:00:00+03:00  280.94
1607  2026-10-17 2026-10-17 17:45:00+03:00  250.01
1712  2026-10-18 2026-10-18 20:00:00+03:00  226.67
1805  2026-10-19 2026-10-

In [9]:
# Exercise 7: peak (Mon-Fri 08:00-20:00 local) vs off-peak
# SQL: SELECT CASE WHEN weekday < 5 AND hour_local >= 8 AND hour_local < 20 THEN 'peak' ELSE 'off-peak' END AS period,
#             COUNT(*), AVG(price)
#      FROM #prices GROUP BY <same CASE>
is_peak = (data_df["weekday"] < 5) & (data_df["hour_local"] >= 8) & (data_df["hour_local"] < 20)
data_df["period"] = np.where(is_peak, "peak", "off-peak")

period_stats = data_df.groupby("period")["price"].agg(["size", "mean"]).reset_index()
print(period_stats)

peak_avg = period_stats.loc[period_stats["period"] == "peak", "mean"].item()
offpeak_avg = period_stats.loc[period_stats["period"] == "off-peak", "mean"].item()
print("spread:", peak_avg - offpeak_avg)

     period  size        mean
0  off-peak  1924   98.791746
1      peak  1056  139.977045
spread: 41.18529909279911


In [10]:
# Exercise 8: hourly profile
# SQL: SELECT hour_local, COUNT(*), AVG(price) FROM #prices GROUP BY hour_local ORDER BY AVG(price)
hourly = data_df.groupby("hour_local")["price"].agg(["size", "mean"]).reset_index()
print(hourly)
print(hourly.nsmallest(1, "mean"))   # cheapest hour
print(hourly.nlargest(1, "mean"))    # priciest hour

    hour_local  size        mean
0            0   124   80.446048
1            1   124   89.086532
2            2   124   88.397823
3            3   128   81.556406
4            4   124   87.344032
5            5   124   85.312661
6            6   124   93.176210
7            7   124  121.506855
8            8   124  132.995645
9            9   124  132.649435
10          10   124  116.358952
11          11   124  104.823952
12          12   124   94.115565
13          13   124   93.930645
14          14   124   92.520161
15          15   124  103.457016
16          16   124  128.367661
17          17   124  153.767742
18          18   124  165.241048
19          19   124  164.219194
20          20   124  154.810000
21          21   124  137.466290
22          22   124  121.600645
23          23   124   99.146694
   hour_local  size       mean
0           0   124  80.446048
    hour_local  size        mean
18          18   124  165.241048


In [11]:
# Exercise 9: 7-day moving average of the daily average
# SQL: SELECT date_local, mean,
#             AVG(mean) OVER (ORDER BY date_local ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS avg_7d
#      FROM #daily
data_df_daily = data_df_daily.sort_values("date_local")
data_df_daily["avg_7d"] = data_df_daily["mean"].rolling(7).mean()
print(data_df_daily[["date_local", "mean", "avg_7d"]])

    date_local        mean      avg_7d
0   2026-10-01   51.558333         NaN
1   2026-10-02   55.554688         NaN
2   2026-10-03   46.502917         NaN
3   2026-10-04   34.540104         NaN
4   2026-10-05    7.166042         NaN
5   2026-10-06   63.851146         NaN
6   2026-10-07  197.742917   65.273735
7   2026-10-08  135.295208   77.236146
8   2026-10-09   28.313021   73.344479
9   2026-10-10   24.750417   70.236979
10  2026-10-11  105.218542   80.333899
11  2026-10-12  151.638542  100.972827
12  2026-10-13  224.457812  123.916637
13  2026-10-14  105.635417  110.758423
14  2026-10-15  201.972708  120.283780
15  2026-10-16  176.837917  141.501622
16  2026-10-17  150.967604  159.532649
17  2026-10-18   98.976979  158.640997
18  2026-10-19  201.837604  165.812292
19  2026-10-20  140.771042  153.857039
20  2026-10-21   66.953646  148.331071
21  2026-10-22   60.508438  128.121890
22  2026-10-23  158.243333  125.465521
23  2026-10-24   27.979479  107.895789
24  2026-10-25    3.01860

In [12]:
# Exercise 10: JOIN a calendar table, weekday vs weekend
# SQL: SELECT c.is_weekend, COUNT(*), AVG(p.price)
#      FROM #prices p LEFT JOIN #calendar c ON c.date_local = p.date_local
#      GROUP BY c.is_weekend
calendar = pd.DataFrame({"date_local": pd.date_range("2026-10-01", "2026-10-31").date})
calendar["is_weekend"] = pd.to_datetime(calendar["date_local"]).dt.day_of_week >= 5

joined = data_df.merge(calendar, on="date_local", how="left")
print(len(joined), joined["is_weekend"].isna().sum())     # 2980 0
print(joined.groupby("is_weekend")["price"].agg(["size", "mean"]))

2980 0
            size        mean
is_weekend                  
False       2112  127.098608
True         868   80.021682
